# Week 3: The Linear Model

## What you will learn this week

- The **linear model formula**: $\hat{y} = w_1 z_1 + w_2 z_2 + \ldots + w_n z_n + b$
- How to **make predictions** using weights and normalized features
- How to **interpret weights** — what each weight says about the real world
- How **different weight settings produce different models** with different beliefs

We apply the same formula to four very different problems:

1. House Price Prediction
2. Student Exam Score Prediction
3. Email Spam Scoring
4. Energy Consumption Prediction

---

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use('seaborn-v0_8-whitegrid')
matplotlib.rcParams['figure.dpi'] = 110
matplotlib.rcParams['font.size'] = 11

print('Libraries loaded.')

---

## The Linear Model Formula

### Concept

A linear model computes a prediction by taking a **weighted sum** of normalized input features and adding a **bias** (baseline offset):

$$\hat{y} = w_1 z_1 + w_2 z_2 + w_3 z_3 + b$$

Where:
- $z_i$ = normalized (z-scored) feature values  
- $w_i$ = weights — how much each feature matters  
- $b$ = bias — the baseline prediction when all features are average (z=0)  
- $\hat{y}$ = the predicted output

### Interpreting Weights

| Weight value | Meaning |
|---|---|
| Large positive (e.g. 0.8) | Feature strongly pushes prediction up |
| Small positive (e.g. 0.1) | Feature weakly pushes prediction up |
| Near zero (e.g. 0.0) | Feature barely matters |
| Negative (e.g. -0.3) | Feature pushes prediction down |

Because inputs are z-scored, weights are directly comparable — a weight of 0.6 is 3x more impactful than a weight of 0.2.

In [ ]:
# ── Shared utility functions used throughout this notebook ────────────────────

def zscore_transform(raw_values, train_mean, train_std):
    """Normalize a value using training set mean and std."""
    return (raw_values - train_mean) / train_std


def linear_predict(z_features, weights, bias):
    """Compute ŷ = sum(w_i * z_i) + b and return both ŷ and contributions."""
    contributions = np.array(weights) * np.array(z_features)
    y_hat = contributions.sum() + bias
    return y_hat, contributions


def print_prediction_breakdown(feat_names, z_vals, weights, bias, y_hat, contributions):
    """Pretty-print each feature's contribution."""
    print(f'  {"Feature":20}  {"z-value":>9}  {"Weight":>8}  {"Contribution":>13}')
    print('  ' + '-' * 55)
    for name, z, w, c in zip(feat_names, z_vals, weights, contributions):
        print(f'  {name:20}  {z:>+9.3f}  {w:>8.3f}  {c:>+13.4f}')
    print('  ' + '-' * 55)
    print(f'  {"Bias":20}  {"":>9}  {bias:>8.3f}  {bias:>+13.4f}')
    print(f'  {"PREDICTION (ŷ)":20}  {"":>9}  {"":>8}  {y_hat:>+13.4f}')


def stacked_bar_plot(ax, feat_names, contributions, bias, y_hat, title, unit=''):
    """Draw a stacked bar showing each feature's contribution to the prediction."""
    all_terms = list(contributions) + [bias]
    all_labels = list(feat_names) + ['Bias']
    colors_pos = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2', '#64B5CD']
    colors_neg = ['#A8C4E0', '#F0C090', '#A8D4B0', '#E0A0A0', '#C0B0D8', '#A0C8D8']

    cumulative = 0.0
    for i, (val, label) in enumerate(zip(all_terms, all_labels)):
        color = colors_pos[i % len(colors_pos)] if val >= 0 else colors_neg[i % len(colors_neg)]
        ax.bar(0, val, bottom=cumulative, width=0.5, color=color,
               edgecolor='white', linewidth=1.2, label=f'{label}: {val:+.3f}')
        if abs(val) > 0.03:
            ax.text(0, cumulative + val / 2, f'{val:+.3f}',
                    ha='center', va='center', fontsize=9, color='white', fontweight='bold')
        cumulative += val

    ax.axhline(0, color='black', linewidth=1.2)
    ax.axhline(y_hat, color='red', linewidth=2.0, linestyle='--', alpha=0.8)
    ax.set_title(title, fontweight='bold')
    ax.set_xticks([])
    ax.set_ylabel(f'Contribution to prediction {unit}')
    ax.legend(loc='lower right', fontsize=8)
    ax.text(0.35, y_hat, f' ŷ = {y_hat:.3f}', va='center', color='red',
            fontsize=9, fontweight='bold')


print('Utility functions defined.')

---

## Application 1: House Price Prediction

### Formula

$$\hat{y}_{\text{price}} = 0.6 \cdot z_{\text{size}} + 0.3 \cdot z_{\text{rooms}} + (-0.2) \cdot z_{\text{age}} + 0.5$$

### Interpreting the weights

- **Size (0.6)**: The most important factor. Every standard deviation increase in size adds 0.6 units to the predicted price.
- **Rooms (0.3)**: Matters about half as much as size. More rooms = higher price, but this effect is smaller.
- **Age (−0.2)**: Negative weight — older houses are cheaper. The magnitude is small, so age matters less than size or rooms.
- **Bias (0.5)**: Even an exactly-average house (all z=0) gets a predicted score of 0.5 — a slight upward baseline.

In [ ]:
# ── Application 1: House Price Prediction ────────────────────────────────────

# Training data statistics (used to normalize new houses)
house_train_stats = {
    'size':  {'mean': 1800, 'std': 400},   # sqft
    'rooms': {'mean': 3.5,  'std': 1.2},
    'age':   {'mean': 20,   'std': 12},    # years since built
}

# Model weights
w_house = [0.6, 0.3, -0.2]
b_house = 0.5
feat_names_house = ['z_size (sqft)', 'z_rooms', 'z_age (years)']

# New house to predict: 2200 sqft, 4 bedrooms, 8 years old
new_house_raw = {'size': 2200, 'rooms': 4, 'age': 8}

print('=== APPLICATION 1: HOUSE PRICE PREDICTION ===')
print(f'  Formula: ŷ = {w_house[0]}·z_size + {w_house[1]}·z_rooms + ({w_house[2]})·z_age + {b_house}')
print(f'\n  New house: {new_house_raw}')
print(f'\n  Step 1 — Z-score each feature:')
z_house = []
for feat, raw in new_house_raw.items():
    stats = house_train_stats[feat]
    z = (raw - stats['mean']) / stats['std']
    z_house.append(z)
    print(f'    z_{feat} = ({raw} - {stats["mean"]}) / {stats["std"]} = {z:+.4f}')

print(f'\n  Step 2 — Compute weighted sum:')
y_hat_house, contributions_house = linear_predict(z_house, w_house, b_house)
print_prediction_breakdown(feat_names_house, z_house, w_house, b_house,
                            y_hat_house, contributions_house)

print(f'\n  Interpretation: This house scores {y_hat_house:.3f} (above 0.5 baseline).')
print(f'  Size contributed most (+{contributions_house[0]:.3f}); rooms also positive (+{contributions_house[1]:.3f}).')
print(f'  Being only 8 years old gives a boost (age=-0.2, z_age<0 → positive contribution {contributions_house[2]:+.3f}).')

In [ ]:
# ── House: Visualize multiple houses and their feature contributions ───────────
# Define several houses to compare
houses_to_compare = [
    {'label': 'Small Old\n(1200sqft, 2BR, 40yr)',
     'raw': [1200, 2, 40]},
    {'label': 'Average\n(1800sqft, 4BR, 20yr)',
     'raw': [1800, 4, 20]},
    {'label': 'Large New\n(2200sqft, 4BR, 8yr)',
     'raw': [2200, 4, 8]},
    {'label': 'Huge Luxury\n(3000sqft, 5BR, 3yr)',
     'raw': [3000, 5, 3]},
]
feat_keys = ['size', 'rooms', 'age']

# Compute z-scores and predictions for each
results_house = []
for h in houses_to_compare:
    z_vals = []
    for feat, raw in zip(feat_keys, h['raw']):
        stats = house_train_stats[feat]
        z_vals.append((raw - stats['mean']) / stats['std'])
    y_hat, contribs = linear_predict(z_vals, w_house, b_house)
    results_house.append({'label': h['label'], 'z': z_vals, 'y_hat': y_hat, 'contribs': contribs})

# Plot: stacked bar for each house
fig, axes = plt.subplots(1, 4, figsize=(16, 6))
fig.suptitle('Application 1 — House Price: Feature Contributions to Prediction',
             fontsize=13, fontweight='bold')

for ax, res in zip(axes, results_house):
    stacked_bar_plot(ax, feat_names_house, res['contribs'], b_house,
                     res['y_hat'], res['label'])

plt.tight_layout()
plt.show()

print('Each stacked bar shows how much each feature pushed the prediction up or down.')
print('Larger houses score higher; older houses score lower.')

### Key Insight

The "Small Old" house has size and age both working against it — z_size is negative (below average) and z_age is positive but multiplied by a negative weight, so age also hurts. The "Huge Luxury" house gets large positive contributions from size, rooms, and a bonus from being very new. The stacked bar makes the model's reasoning fully transparent.

---

## Application 2: Student Exam Score Prediction

### Formula

$$\hat{y}_{\text{score}} = 0.5 \cdot z_{\text{study}} + 0.3 \cdot z_{\text{attend}} + 0.4 \cdot z_{\text{prev}} + 0.0$$

### Interpreting the weights

- **Study hours (0.5)**: The most important input. Effort pays off most.
- **Previous score (0.4)**: Past performance is nearly as important — prior knowledge carries over.
- **Attendance (0.3)**: Important but less so than effort or prior knowledge.
- **Bias (0.0)**: A student who is exactly average in all three features scores exactly 0 (normalized scale).

In [ ]:
# ── Application 2: Student Exam Score Prediction ──────────────────────────────

student_train_stats = {
    'study_hrs': {'mean': 2.5,  'std': 1.5},
    'attendance': {'mean': 78.0, 'std': 15.0},
    'prev_score': {'mean': 70.0, 'std': 12.0},
}

w_student = [0.5, 0.3, 0.4]
b_student = 0.0
feat_names_student = ['z_study_hrs', 'z_attendance', 'z_prev_score']

# Three student profiles
student_profiles = [
    {'label': 'Diligent\n(4hr, 95%, prev=85)', 'raw': [4.0, 95, 85]},
    {'label': 'Average\n(2.5hr, 78%, prev=70)', 'raw': [2.5, 78, 70]},
    {'label': 'Struggling\n(1hr, 50%, prev=58)', 'raw': [1.0, 50, 58]},
]
feat_keys_s = ['study_hrs', 'attendance', 'prev_score']

print('=== APPLICATION 2: STUDENT EXAM SCORE PREDICTION ===')
print(f'  Formula: ŷ = {w_student[0]}·z_study + {w_student[1]}·z_attend + {w_student[2]}·z_prev + {b_student}')

results_student = []
for sp in student_profiles:
    print(f'\n  --- {sp["label"].replace(chr(10), " ")} ---')
    z_vals = []
    for feat, raw in zip(feat_keys_s, sp['raw']):
        stats = student_train_stats[feat]
        z = (raw - stats['mean']) / stats['std']
        z_vals.append(z)
        print(f'    z_{feat} = ({raw} - {stats["mean"]}) / {stats["std"]} = {z:+.3f}')
    y_hat, contribs = linear_predict(z_vals, w_student, b_student)
    print_prediction_breakdown(feat_names_student, z_vals, w_student, b_student, y_hat, contribs)
    results_student.append({'label': sp['label'], 'z': z_vals, 'y_hat': y_hat, 'contribs': contribs})

In [ ]:
# ── Student: Visualization ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 6))
fig.suptitle('Application 2 — Student Score: Feature Contributions for Three Profiles',
             fontsize=13, fontweight='bold')

for ax, res in zip(axes, results_student):
    stacked_bar_plot(ax, feat_names_student, res['contribs'], b_student,
                     res['y_hat'], res['label'])

plt.tight_layout()
plt.show()

# Also show a comparison bar chart
fig2, ax = plt.subplots(figsize=(8, 5))
labels = [r['label'].replace('\n', ' ') for r in results_student]
y_hats = [r['y_hat'] for r in results_student]
colors_s = ['#55A868', '#4C72B0', '#C44E52']
bars = ax.bar(range(len(labels)), y_hats, color=colors_s, alpha=0.8, width=0.5)
ax.axhline(0, color='black', linewidth=1.2, linestyle='--', alpha=0.6)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=10)
ax.set_ylabel('Predicted Score (normalized)')
ax.set_title('Predicted Score Comparison Across Student Profiles', fontweight='bold')
ax.bar_label(bars, fmt='%.3f', padding=4)
plt.tight_layout()
plt.show()

### Key Insight

The diligent student has all three features above average, so all three contributions are positive — the model expects a good score. The struggling student has all features below average, so all contributions are negative. The weights tell the story: study hours matter most (0.5), so effort is the strongest lever a student has.

---

## Application 3: Email Spam Score

### Formula

$$\hat{y}_{\text{spam}} = 0.4 \cdot z_{\text{free}} + 0.35 \cdot z_{\text{caps}} + 0.3 \cdot z_{\text{links}} + (-0.1)$$

### Interpreting the weights + decision threshold

- **FREE count (0.4)**: Highest weight — this word is the strongest spam signal.
- **CAPS ratio (0.35)**: Nearly as important — shouting indicates spam.
- **Link count (0.3)**: Multiple links suggest phishing or spam.
- **Bias (−0.1)**: A slight default assumption toward not-spam, reducing false positives.
- **Decision rule**: If $\hat{y} > 0.5$ → mark as spam

In [ ]:
# ── Application 3: Email Spam Score ──────────────────────────────────────────

spam_train_stats = {
    'free_count': {'mean': 1.0, 'std': 1.5},
    'caps_ratio':  {'mean': 0.1, 'std': 0.12},
    'link_count':  {'mean': 1.5, 'std': 2.0},
}

w_spam = [0.4, 0.35, 0.3]
b_spam = -0.1
feat_names_spam = ['z_free_count', 'z_caps_ratio', 'z_link_count']
SPAM_THRESHOLD = 0.5

# Three emails
email_profiles = [
    {'label': 'Obvious Spam\n(FREE=5, CAPS=45%, links=8)',
     'raw': [5, 0.45, 8],
     'text': 'FREE IPHONE!!! Click NOW!!!'},
    {'label': 'Legitimate\n(FREE=0, CAPS=2%, links=1)',
     'raw': [0, 0.02, 1],
     'text': 'Meeting at 3pm in conference room B'},
    {'label': 'Borderline\n(FREE=2, CAPS=15%, links=3)',
     'raw': [2, 0.15, 3],
     'text': 'Free trial - click to subscribe today'},
]
feat_keys_spam = ['free_count', 'caps_ratio', 'link_count']

print('=== APPLICATION 3: EMAIL SPAM SCORE ===')
print(f'  Formula: ŷ = {w_spam[0]}·z_free + {w_spam[1]}·z_caps + {w_spam[2]}·z_links + ({b_spam})')
print(f'  Decision: ŷ > {SPAM_THRESHOLD} → SPAM')

results_spam = []
for ep in email_profiles:
    print(f'\n  --- {ep["label"].replace(chr(10), " ")} ---')
    print(f'  Email text: "{ep["text"]}"')
    z_vals = []
    for feat, raw in zip(feat_keys_spam, ep['raw']):
        stats = spam_train_stats[feat]
        z = (raw - stats['mean']) / stats['std']
        z_vals.append(z)
        print(f'    z_{feat} = ({raw} - {stats["mean"]}) / {stats["std"]} = {z:+.3f}')
    y_hat, contribs = linear_predict(z_vals, w_spam, b_spam)
    decision = 'SPAM' if y_hat > SPAM_THRESHOLD else 'NOT SPAM'
    print_prediction_breakdown(feat_names_spam, z_vals, w_spam, b_spam, y_hat, contribs)
    print(f'  Decision: {y_hat:.3f} {">": } {SPAM_THRESHOLD} → {decision}')
    results_spam.append({'label': ep['label'], 'z': z_vals, 'y_hat': y_hat,
                          'contribs': contribs, 'decision': decision})

In [ ]:
# ── Spam: Visualization ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 6))
fig.suptitle('Application 3 — Email Spam: Feature Contributions and Decision Threshold',
             fontsize=13, fontweight='bold')

for ax, res in zip(axes, results_spam):
    stacked_bar_plot(ax, feat_names_spam, res['contribs'], b_spam,
                     res['y_hat'], res['label'])
    # Show the threshold line
    ax.axhline(SPAM_THRESHOLD, color='orange', linewidth=2.0, linestyle='-.',
               label=f'Threshold = {SPAM_THRESHOLD}')
    ax.legend(loc='lower right', fontsize=8)
    decision_color = '#C44E52' if res['decision'] == 'SPAM' else '#55A868'
    ax.text(0.5, 0.98, res['decision'], transform=ax.transAxes,
            ha='center', va='top', fontsize=14, fontweight='bold', color=decision_color)

plt.tight_layout()
plt.show()

print('Orange dashed line = spam threshold (0.5).')
print('If the red prediction line is above orange → SPAM.')

### Key Insight

The borderline email scores just below the threshold — it has "Free" and a few links but low caps usage. By adjusting the threshold (e.g., from 0.5 to 0.3), the model becomes more aggressive about catching spam but will also start flagging more legitimate emails. The threshold is a design choice, not part of the model weights.

---

## Application 4: Building Energy Consumption

### Formula

$$\hat{y}_{\text{energy}} = 0.3 \cdot z_{\text{temp}} + 0.5 \cdot z_{\text{occ}} + 0.2 \cdot z_{\text{hour}} + 0.4$$

### Interpreting the weights

- **Occupancy (0.5)**: The biggest driver. More people = more lights, computers, HVAC load.
- **Temperature (0.3)**: Hot outside means more air conditioning.
- **Hour of day (0.2)**: Captures time-of-day patterns (peak hours use more energy).
- **Bias (0.4)**: Even with all features at average, the building still uses baseline energy (servers, standby systems).

In [ ]:
# ── Application 4: Energy Consumption ────────────────────────────────────────

energy_train_stats = {
    'temperature': {'mean': 22.0, 'std': 6.0},    # Celsius
    'occupancy':   {'mean': 150,  'std': 80.0},    # number of people
    'hour_of_day': {'mean': 12.0, 'std': 5.0},     # 0–23
}

w_energy = [0.3, 0.5, 0.2]
b_energy = 0.4
feat_names_energy = ['z_temperature', 'z_occupancy', 'z_hour']

# Four scenarios to predict
energy_scenarios = [
    {'label': 'Hot Summer\nPeak Hours\n(35°C, 250ppl, 14:00)',  'raw': [35, 250, 14]},
    {'label': 'Morning\nStartup\n(20°C, 50ppl, 08:00)',          'raw': [20,  50,  8]},
    {'label': 'Overnight\nEmpty\n(18°C, 5ppl, 02:00)',           'raw': [18,   5,  2]},
    {'label': 'Full Office\nMild Day\n(22°C, 200ppl, 10:00)',    'raw': [22, 200, 10]},
]
feat_keys_e = ['temperature', 'occupancy', 'hour_of_day']

print('=== APPLICATION 4: ENERGY CONSUMPTION ===')
print(f'  Formula: ŷ = {w_energy[0]}·z_temp + {w_energy[1]}·z_occ + {w_energy[2]}·z_hour + {b_energy}')

results_energy = []
for sc in energy_scenarios:
    print(f'\n  Scenario: {sc["label"].replace(chr(10)," ")}')
    z_vals = []
    for feat, raw in zip(feat_keys_e, sc['raw']):
        stats = energy_train_stats[feat]
        z = (raw - stats['mean']) / stats['std']
        z_vals.append(z)
    y_hat, contribs = linear_predict(z_vals, w_energy, b_energy)
    print_prediction_breakdown(feat_names_energy, z_vals, w_energy, b_energy, y_hat, contribs)
    results_energy.append({'label': sc['label'], 'z': z_vals, 'y_hat': y_hat, 'contribs': contribs})

In [ ]:
# ── Energy: Visualization ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 6))
fig.suptitle('Application 4 — Energy Consumption: Feature Contributions by Scenario',
             fontsize=13, fontweight='bold')

for ax, res in zip(axes, results_energy):
    stacked_bar_plot(ax, feat_names_energy, res['contribs'], b_energy,
                     res['y_hat'], res['label'], unit='(normalized)')

plt.tight_layout()
plt.show()

# Summary bar chart
fig2, ax = plt.subplots(figsize=(9, 5))
labels_e = [r['label'].replace('\n', ' ') for r in results_energy]
y_hats_e = [r['y_hat'] for r in results_energy]
colors_e = ['#C44E52', '#DD8452', '#4C72B0', '#55A868']
bars_e = ax.bar(range(len(labels_e)), y_hats_e, color=colors_e, alpha=0.8, width=0.5)
ax.axhline(b_energy, color='gray', linewidth=1.2, linestyle='--', alpha=0.7,
           label=f'Bias baseline = {b_energy}')
ax.set_xticks(range(len(labels_e)))
ax.set_xticklabels(labels_e, rotation=10, ha='right')
ax.set_ylabel('Predicted Energy (normalized)')
ax.set_title('Predicted Energy Across Scenarios', fontweight='bold')
ax.bar_label(bars_e, fmt='%.3f', padding=4)
ax.legend()
plt.tight_layout()
plt.show()

### Key Insight

The overnight scenario (almost empty building at 2am) scores lowest — all features push the prediction down. The hot summer peak-hours scenario scores highest. Occupancy dominates because it has the largest weight (0.5) — and in the full-office scenario, even though temperature is mild, high occupancy alone is enough to drive consumption up significantly.

---

## Weight Comparison: Same Person, Three Different Models

### Concept

The weights in a model encode its **beliefs** about the world. Two models trained on different data (or with different assumptions) will produce different weights — and therefore different predictions for the same input.

Below we take the same student and run them through three models:

- **Model A** — Study-focused: believes effort matters most
- **Model B** — Talent-focused: believes prior score matters most
- **Model C** — Balanced: weights all features roughly equally

All three models see the SAME student. The different weights produce different predictions — and different recommendations for the student.

In [ ]:
# ── Three Models: Same Student, Different Weights ─────────────────────────────

# Same student for all three models
same_student_raw = [3.5, 88, 75]   # study_hrs=3.5, attendance=88%, prev_score=75

# Normalize using training stats (from Application 2)
z_same = []
for feat, raw in zip(['study_hrs', 'attendance', 'prev_score'], same_student_raw):
    stats = student_train_stats[feat]
    z = (raw - stats['mean']) / stats['std']
    z_same.append(z)

print(f'Same student: study={same_student_raw[0]}hr, attendance={same_student_raw[1]}%, prev={same_student_raw[2]}')
print(f'Z-scores: z_study={z_same[0]:+.3f}, z_attend={z_same[1]:+.3f}, z_prev={z_same[2]:+.3f}')

# Three different models (same features, different weights)
model_configs = [
    {'name': 'Model A\n(Study-Focused)',   'weights': [0.7,  0.1,  0.2],  'bias': 0.0,
     'description': 'Believes effort is everything'},
    {'name': 'Model B\n(Talent-Focused)',  'weights': [0.1,  0.1,  0.8],  'bias': 0.0,
     'description': 'Believes prior ability dominates'},
    {'name': 'Model C\n(Balanced)',        'weights': [0.35, 0.3,  0.35], 'bias': 0.0,
     'description': 'All factors equally important'},
]

print(f'\n=== THREE MODELS, SAME STUDENT ===')
print(f'  {"Model":25}  {"w_study":>9}  {"w_attend":>9}  {"w_prev":>9}  {"Prediction":>12}')
print('  ' + '-' * 68)

results_comparison = []
for mc in model_configs:
    y_hat, contribs = linear_predict(z_same, mc['weights'], mc['bias'])
    print(f'  {mc["name"].replace(chr(10), " "):25}  '
          f'{mc["weights"][0]:>9.2f}  {mc["weights"][1]:>9.2f}  '
          f'{mc["weights"][2]:>9.2f}  {y_hat:>+12.4f}')
    results_comparison.append({'mc': mc, 'y_hat': y_hat, 'contribs': contribs})

print(f'\nAll three models see the SAME student but predict differently because')
print(f'they have different beliefs about what drives exam success.')

In [ ]:
# ── Comparison: Side-by-side visualization ───────────────────────────────────
fig = plt.figure(figsize=(16, 10))
gs  = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)
fig.suptitle('Three Models, Same Student: How Different Weights Produce Different Beliefs',
             fontsize=13, fontweight='bold')

colors_models = ['#4C72B0', '#DD8452', '#55A868']

# Top row — stacked bars showing each model's reasoning
for col, (res, color) in enumerate(zip(results_comparison, colors_models)):
    ax = fig.add_subplot(gs[0, col])
    stacked_bar_plot(ax, feat_names_student, res['contribs'],
                     res['mc']['bias'], res['y_hat'],
                     res['mc']['name'] + f"\n'{res['mc']['description']}'")

# Bottom row — weight profiles and prediction comparison
ax_weights = fig.add_subplot(gs[1, :2])
model_names = [r['mc']['name'].replace('\n', ' ') for r in results_comparison]
w_study  = [r['mc']['weights'][0] for r in results_comparison]
w_attend = [r['mc']['weights'][1] for r in results_comparison]
w_prev   = [r['mc']['weights'][2] for r in results_comparison]

x_m = np.arange(len(model_names))
width_m = 0.25
bars_s = ax_weights.bar(x_m - width_m, w_study,  width_m, label='w_study',  color='#4C72B0', alpha=0.85)
bars_a = ax_weights.bar(x_m,           w_attend, width_m, label='w_attend', color='#DD8452', alpha=0.85)
bars_p = ax_weights.bar(x_m + width_m, w_prev,   width_m, label='w_prev',   color='#55A868', alpha=0.85)
ax_weights.set_xticks(x_m)
ax_weights.set_xticklabels(model_names)
ax_weights.set_ylabel('Weight Value')
ax_weights.set_title('Weight Profiles for Each Model', fontweight='bold')
ax_weights.legend()
ax_weights.bar_label(bars_s, fmt='%.2f', padding=2, fontsize=8)
ax_weights.bar_label(bars_a, fmt='%.2f', padding=2, fontsize=8)
ax_weights.bar_label(bars_p, fmt='%.2f', padding=2, fontsize=8)

ax_preds = fig.add_subplot(gs[1, 2])
preds_comp = [r['y_hat'] for r in results_comparison]
bars_comp  = ax_preds.bar(range(len(preds_comp)), preds_comp,
                           color=colors_models, alpha=0.85, width=0.5)
ax_preds.axhline(0, color='black', linewidth=1.2, linestyle='--', alpha=0.5)
ax_preds.set_xticks(range(len(model_names)))
ax_preds.set_xticklabels([n.split(' ')[0] + ' ' + n.split(' ')[1] for n in model_names])
ax_preds.set_ylabel('Prediction (normalized)')
ax_preds.set_title('Final Predictions\n(same student)', fontweight='bold')
ax_preds.bar_label(bars_comp, fmt='%.3f', padding=4)

plt.show()

In [ ]:
# ── What the different predictions mean ──────────────────────────────────────
print('=== INTERPRETATION ===')
print(f'  Student profile: 3.5 study hrs/day, 88% attendance, prev score = 75')
print(f'  z_study = {z_same[0]:+.3f} (above average effort)')
print(f'  z_attend = {z_same[1]:+.3f} (above average attendance)')
print(f'  z_prev = {z_same[2]:+.3f} (slightly above average prior score)')
print()
for res in results_comparison:
    mc = res['mc']
    y  = res['y_hat']
    print(f'  {mc["name"].replace(chr(10), " "):28} → ŷ = {y:+.4f}')
    print(f'  Belief: "{mc["description"]}"')
    dominant = feat_names_student[np.argmax(mc['weights'])]
    print(f'  This model says: the dominant factor for this student is {dominant}')
    print()

print('Which model is right? That depends on the training data.')
print('The model trained on the most relevant historical data will have learned')
print('the weights that minimize prediction error — and those weights are the truth.')

### Key Insight

Model A rewards this student most because they study hard (z_study = +0.67) and Model A's study weight is 0.7. Model B barely rewards them because prior score is only slightly above average (z_prev = +0.42) and that's what Model B cares about most. Model C gives an in-between score.

**The weights ARE the model's beliefs.** Two data scientists training on different schools' data might end up with different weights — and that's completely valid, as long as each model minimizes error on its own data.

---

## Summary Visualization: All Four Applications Side by Side

Let's look at one example from each application to see how the same formula works across completely different domains.

In [ ]:
# ── Summary: Weight magnitudes across all 4 applications ─────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('Weight Magnitudes Across All Four Applications\n'
             '(Larger bar = feature has more influence on prediction)',
             fontsize=12, fontweight='bold')

app_summaries = [
    {
        'title': 'House Price',
        'features': ['Size', 'Rooms', 'Age'],
        'weights': [0.6, 0.3, -0.2],
        'bias': 0.5,
        'color': '#4C72B0',
    },
    {
        'title': 'Student Score',
        'features': ['Study Hrs', 'Attendance', 'Prev Score'],
        'weights': [0.5, 0.3, 0.4],
        'bias': 0.0,
        'color': '#DD8452',
    },
    {
        'title': 'Email Spam',
        'features': ['FREE count', 'CAPS ratio', 'Link count'],
        'weights': [0.4, 0.35, 0.3],
        'bias': -0.1,
        'color': '#C44E52',
    },
    {
        'title': 'Energy Use',
        'features': ['Temperature', 'Occupancy', 'Hour'],
        'weights': [0.3, 0.5, 0.2],
        'bias': 0.4,
        'color': '#55A868',
    },
]

for ax, app in zip(axes, app_summaries):
    w = app['weights']
    features = app['features'] + ['Bias']
    values   = w + [app['bias']]
    bar_colors = [app['color'] if v >= 0 else '#AAAAAA' for v in values]
    bars = ax.barh(features, values, color=bar_colors, alpha=0.8, edgecolor='white')
    ax.axvline(0, color='black', linewidth=1.2)
    ax.set_title(app['title'], fontweight='bold')
    ax.set_xlabel('Weight value')
    ax.set_xlim(-0.7, 0.75)
    ax.bar_label(bars, fmt='%.2f', padding=4, fontsize=9)

plt.tight_layout()
plt.show()

print('One formula. Four problems. The only thing that changes is the weights.')

### Key Insight

All four applications use exactly the same formula: `ŷ = w₁z₁ + w₂z₂ + w₃z₃ + b`. The weights look different because the real world is different:
- In houses, size dominates
- In student scores, study hours dominate
- In spam, FREE-word count dominates
- In energy, occupancy dominates

The job of machine learning is to automatically discover these weights from data — rather than guessing them by hand as we did here.

---

## Week 3 Checkpoint

After completing this notebook you should be able to:

1. **Write and evaluate the linear model formula** `ŷ = w₁z₁ + w₂z₂ + ... + b` for any set of features, including z-scoring inputs and computing each feature's individual contribution to the final prediction.

2. **Interpret model weights**: larger absolute value = more influential feature; positive weight = feature pushes prediction up; negative weight = feature pushes prediction down; bias = baseline prediction when all features are exactly average.

3. **Use the stacked contribution bar chart** to explain any prediction: read off each feature's contribution, verify they sum to the final ŷ, and identify which feature is most responsible for a high or low prediction.

4. **Explain why different weight sets produce different models**: weights encode beliefs about the world learned from training data, so a model trained on university A's data will learn different weights than one trained on university B's data — even though they use the same formula and the same features.